# Modellierungsseminar Sommer 2026
## Cycle Planning for workforce scheduling

In [22]:
# import required packages
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from dataclasses import dataclass
import src.Shift as Shift # tailor-made data type for shift definitions

### inputs and parameters

In [23]:

# basic inputs and parameters

Weekdays = range(1,8)  # results in 1,...,7 => let 1 be Monday and 7 be Sunday
Shifts = ["frueh", "spaet", "nacht", "frei"] # including free shifts
WorkShifts = ["frueh", "spaet", "nacht"] # excluding free shifts

# improvements outstanding:
    # use files for parameter input
        # shift definitions
        # available staff
        # user objectives: weighted priorities
    

# more relevant for more complex models going forward
MAX_CYCLE_WEEKS = int(52/4)  # in future this shall be user's input => too long snakes do not help to ensure "fair" distribution of shifts
DICT_WEEKDAYS = {'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6,'Sun':7,
                 'Monday':1,'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7,
                 'Mo':1,'Tu':2,'We':3,'Th':4,'Fr':5,'Sa':6,'Su':7,
                 '1':1,'2':2,'3':3,'4':4,'5':5,'6':6,'7':7

}


In [24]:
# read input data
# shift set

def readShiftSet(filename: str, mySep: str=";") -> pd.DataFrame:
    input_data = pd.read_csv(filename, sep=mySep, dtype=str) # import all values as string as first step 
    return input_data
    # potentially add data cleaning steps

folderpath = "input/"
filename = "input_ShiftDataSet_Pesch.csv"

data_shiftSet = readShiftSet(folderpath + filename)
data_shiftSet["isWorkShift"] = data_shiftSet["isWorkShift"].astype(int).astype(bool)
#print(data_shiftSet)
Shifts = list(data_shiftSet["shift_ID"]) # including free shifts
WorkShifts = list(data_shiftSet[data_shiftSet["isWorkShift"]]["shift_ID"]) # excluding free shifts

#print(Shifts)
#print(data_shiftSet["shift_weekdays"])



### modelling

In [25]:
# modelling

m = gp.Model("SnakeBuilding_simple")

# variables:
# x[s, d, sh] = 1, when snake s is working in shift sh on day d
x = m.addVars(MAX_CYCLE_WEEKS, Weekdays, Shifts, vtype=GRB.BINARY, name="x")

# active[s] = 1, when snake s is used
active = m.addVars(MAX_CYCLE_WEEKS, vtype=GRB.BINARY, name="active") # all other snakes are used as placeholders but not necessarily get activated


### conditions:

#### condition c01
_(idea is to use a unique ID for each condition for better reference)_

each shift has to be covered on each day

$$\sum_{s=1}^{n}{x_{s,d,w}} >= 1    \forall d \in D, \forall w \in W$$

$x_{s,d,w} = 1$, when snake s is working in work shift w on day d

$x: $ binary variable, 
$s: $ snake number, 
$d: $ weekday, 
$ws: $ work shift


In [26]:
# SUBJECT TO:

# 1. each shift has to be covered on each day
for d in Weekdays:
    for ws in WorkShifts:
        m.addConstr(gp.quicksum(x[s, d, ws] for s in range(MAX_CYCLE_WEEKS)) >= 1,
                    name=f"Cover_day{d}_{ws}")


####

#### condition c02

In [27]:
# 2. each snake can have at most one shift per day
for s in range(MAX_CYCLE_WEEKS):
    for d in Weekdays:
        m.addConstr(gp.quicksum(x[s, d, sh] for sh in Shifts) == active[s],
                    name=f"OneShiftPerDay_s{s}_d{d}")


#### condition c03

In [28]:

# 3) at max 5 consecutive working days (ensure time for resting)
for s in range(MAX_CYCLE_WEEKS):
    for start in range(1, 7-5+1):  
        m.addConstr(
            gp.quicksum(x[s, d, sh] for d in range(start, start + 6)
                        for sh in WorkShifts) <= 5,
            name=f"Max5Work_s{s}_start{start}"
        )



### objective

In [29]:
# set objective function: minimize number of active snakes
m.setObjective(gp.quicksum(active[s] for s in range(MAX_CYCLE_WEEKS)), GRB.MINIMIZE)

# improvements outstanding:
    # add various weighted objectives

#run optimizer
m.optimize()



Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 PRO 250 w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Academic license 2806446 - for non-commercial use only - registered to di___@student.uni-siegen.de
Optimize a model with 145 rows, 468 columns and 1534 nonzeros (Min)
Model fingerprint: 0x86e18cee
Model has 13 linear objective coefficients
Variable types: 0 continuous, 468 integer (468 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Found heuristic solution: objective 5.0000000
Presolve removed 30 rows and 143 columns
Presolve time: 0.00s
Presolved: 115 rows, 325 columns, 1014 nonzeros
Variable types: 0 continuous, 325 integer (325 binary)

Root relaxation: objective 4.000000e+00, 194 iterations, 0.00 seconds (0.0

### results

In [30]:
# output (raw version, to be improved for better readability)
    # improvements outstanding: 
        # write results in file
        # create a shift overview per staff member

if m.status == GRB.OPTIMAL:
    print("\nminimum number of cycle weeks:", int(m.objVal))
    for s in range(MAX_CYCLE_WEEKS):
        if active[s].X == 1:
            print(f"\ncycle week {s+1}:")
            for d in Weekdays:
                for sh in Shifts:
                    if x[s, d, sh].X == 1:
                        print(f"  day {d}: {sh}")



minimum number of cycle weeks: 5

cycle week 3:
  day 1: [00day000week]
  day 2: [00000freeday]
  day 3: [nightweekend]
  day 4: [night000week]
  day 5: [00dayweekend]
  day 6: [00dayweekend]
  day 7: [00dayweekend]

cycle week 5:
  day 1: [00000freeday]
  day 2: [00day000week]
  day 3: [night000week]
  day 4: [00day000week]
  day 5: [00000freeday]
  day 6: [00day000week]
  day 7: [00000freeday]

cycle week 6:
  day 1: [00dayweekend]
  day 2: [night000week]
  day 3: [00day000week]
  day 4: [00dayweekend]
  day 5: [night000week]
  day 6: [00000freeday]
  day 7: [00day000week]

cycle week 10:
  day 1: [nightweekend]
  day 2: [00dayweekend]
  day 3: [00dayweekend]
  day 4: [00000freeday]
  day 5: [nightweekend]
  day 6: [nightweekend]
  day 7: [night000week]

cycle week 12:
  day 1: [night000week]
  day 2: [nightweekend]
  day 3: [00000freeday]
  day 4: [nightweekend]
  day 5: [00day000week]
  day 6: [night000week]
  day 7: [nightweekend]
